## Post-Run Pilot Audit
Analyzing the paired results to surface measurement artifacts and data quality issues before scaling to the full experiment. From the first 100Q run (seed=42) there were 3 findings:
1. `max_tokens` cap: questions where the model was cut off mid-reasoning
2. Loader bug: questions with numeric gold keys that can never match a letter prediction
3. verbosity pattern: wrong answers use significantly more tokens than correct ones

In [1]:
import json
import statistics
import pathlib

results_path = pathlib.Path("../experiments/results/poc_arc_mlx_mistral7b_100.json")

with open(results_path) as f:
    data = json.load(f)

print(f"Loaded {len(data)} paired records")

Loaded 100 paired records


### Discordant Questions
McNemar off-diagonal
* these are the only questions that matter for the McNemar test
* base_only -> repeat hurt
* repeat_only -> repeat helps

In [2]:
base_only   = [r for r in data if     r["base_correct"] and not r["rep_correct"]]
repeat_only = [r for r in data if not r["base_correct"] and     r["rep_correct"]]

print(f"Discordant pairs: {len(base_only)} baseline-only (repeat hurt), "
      f"{len(repeat_only)} repeat-only (repeat helped)")
print()

print("BASELINE-ONLY (repeat hurt):")
for r in base_only:
    print(f"  {r['question_id']}")
    print(f"    gold={r['gold']}  base={r['base_predicted']} ✓  rep={r['rep_predicted']} ✗")
    print(f"    tokens: base={r['base_tokens']}  rep={r['rep_tokens']}  "
          f"latency: base={r['base_latency_ms']:.0f}ms  rep={r['rep_latency_ms']:.0f}ms")

print()
print("REPEAT-ONLY (repeat helped):")
for r in repeat_only:
    print(f"  {r['question_id']}")
    print(f"    gold={r['gold']}  base={r['base_predicted']} ✗  rep={r['rep_predicted']} ✓")
    print(f"    tokens: base={r['base_tokens']}  rep={r['rep_tokens']}  "
          f"latency: base={r['base_latency_ms']:.0f}ms  rep={r['rep_latency_ms']:.0f}ms")

Discordant pairs: 5 baseline-only (repeat hurt), 1 repeat-only (repeat helped)

BASELINE-ONLY (repeat hurt):
  Mercury_404638
    gold=D  base=D ✓  rep=A ✗
    tokens: base=75  rep=123  latency: base=4424ms  rep=7006ms
  Mercury_411224
    gold=B  base=B ✓  rep=A ✗
    tokens: base=129  rep=129  latency: base=7267ms  rep=8507ms
  Mercury_416646
    gold=B  base=B ✓  rep=C ✗
    tokens: base=37  rep=35  latency: base=2640ms  rep=3372ms
  Mercury_417465
    gold=C  base=C ✓  rep=A ✗
    tokens: base=62  rep=65  latency: base=3835ms  rep=4541ms
  Mercury_7017710
    gold=C  base=C ✓  rep=A ✗
    tokens: base=14  rep=11  latency: base=1449ms  rep=2002ms

REPEAT-ONLY (repeat helped):
  Mercury_7210210
    gold=C  base=A ✗  rep=C ✓
    tokens: base=33  rep=39  latency: base=2239ms  rep=2928ms


### `max_tokens` Cap
Questions where the model hit the `max_tokens` ceiling mid-generation
* the answer in these cases depends on wherever reasoning happened to be truncated (not on genuinie model preference)
* any discordant result involving a capped question should be treated as a measurement artifact

Fix: raise `max_tokens` to 256 for full experiment run

In [3]:
MAX_TOKENS = 128  # must match the value used in MLXAgentConfig

capped_base = [r for r in data if r["base_tokens"] >= MAX_TOKENS]
capped_rep  = [r for r in data if r["rep_tokens"]  >= MAX_TOKENS]
capped_ids  = {r["question_id"] for r in capped_base + capped_rep}

print(f"Questions hitting the {MAX_TOKENS}-token cap:")
print(f"  Baseline : {len(capped_base)}")
print(f"  Repeat   : {len(capped_rep)}")
print()

for r in sorted(capped_base + capped_rep, key=lambda x: x["question_id"]):
    b_flag = "⚠ CAP" if r["base_tokens"] >= MAX_TOKENS else "     "
    r_flag = "⚠ CAP" if r["rep_tokens"]  >= MAX_TOKENS else "     "
    disc   = "← DISCORDANT" if (r["base_correct"] != r["rep_correct"]) else ""
    print(f"  {r['question_id']:35}  "
          f"base={r['base_tokens']:3}tok {b_flag}  "
          f"rep={r['rep_tokens']:3}tok {r_flag}  {disc}")

# how many discordant pairs are tainted by a cap?
discordant_ids = {r["question_id"] for r in base_only + repeat_only}
tainted = discordant_ids & capped_ids
print(f"\nDiscordant questions also capped: {len(tainted)} → {tainted}")
print("These should be excluded from McNemar analysis.")

Questions hitting the 128-token cap:
  Baseline : 5
  Repeat   : 5

  AKDE&ED_2008_4_35                    base= 75tok        rep=129tok ⚠ CAP  
  MCAS_2005_9_21                       base=129tok ⚠ CAP  rep=129tok ⚠ CAP  
  MCAS_2005_9_21                       base=129tok ⚠ CAP  rep=129tok ⚠ CAP  
  Mercury_188528                       base=129tok ⚠ CAP  rep=  9tok        
  Mercury_407517                       base=100tok        rep=129tok ⚠ CAP  
  Mercury_411224                       base=129tok ⚠ CAP  rep=129tok ⚠ CAP  ← DISCORDANT
  Mercury_411224                       base=129tok ⚠ CAP  rep=129tok ⚠ CAP  ← DISCORDANT
  Mercury_7271758                      base=129tok ⚠ CAP  rep= 15tok        
  NYSEDREGENTS_2008_8_27               base=129tok ⚠ CAP  rep=129tok ⚠ CAP  
  NYSEDREGENTS_2008_8_27               base=129tok ⚠ CAP  rep=129tok ⚠ CAP  

Discordant questions also capped: 1 → {'Mercury_411224'}
These should be excluded from McNemar analysis.


### Loader Bug: Numeric Gold Answer Keys
ARC-Challnege normally uses letter keys (A/B/C/D) but a small number of NYSEDREGENTS questions use numeric keys (1/2/3/4) in the raw dataset.
* the loader is returning these raw values without mapping them to the letters, so the model's prediction can never match
* these questions must be filtered out or resolve hardcode error in evaluator.py 

In [4]:
valid_gold = set("ABCD")
bad_gold = [r for r in data if r["gold"] not in valid_gold]

print(f"Questions with non-letter gold answers: {len(bad_gold)}")
for r in bad_gold:
    print(f"  {r['question_id']:35}  gold='{r['gold']}'  "
          f"base={r['base_predicted']} rep={r['rep_predicted']}  "
          f"(always wrong — letter can never equal digit)")

print("\nFix: in arc_loader.py, map answerKey '1','2','3','4' → 'A','B','C','D'")
print("     or filter these records out during loading.")

# recalculate accuracy excluding bad questions
clean = [r for r in data if r["gold"] in valid_gold]
clean_base_acc = sum(r["base_correct"] for r in clean) / len(clean)
clean_rep_acc  = sum(r["rep_correct"]  for r in clean) / len(clean)
print(f"\nAccuracy on clean subset ({len(clean)} questions):")
print(f"  Baseline : {clean_base_acc:.1%}  ({sum(r['base_correct'] for r in clean)}/{len(clean)})")
print(f"  Repeat   : {clean_rep_acc:.1%}  ({sum(r['rep_correct'] for r in clean)}/{len(clean)})")
print(f"  Δ        : {clean_rep_acc - clean_base_acc:+.1%}")

Questions with non-letter gold answers: 3
  NYSEDREGENTS_2008_8_30               gold='2'  base=B rep=B  (always wrong — letter can never equal digit)
  NYSEDREGENTS_2008_8_27               gold='3'  base=D rep=C  (always wrong — letter can never equal digit)
  NYSEDREGENTS_2008_8_32               gold='1'  base=A rep=A  (always wrong — letter can never equal digit)

Fix: in arc_loader.py, map answerKey '1','2','3','4' → 'A','B','C','D'
     or filter these records out during loading.

Accuracy on clean subset (97 questions):
  Baseline : 69.1%  (67/97)
  Repeat   : 64.9%  (63/97)
  Δ        : -4.1%


### Verbosity Pattern
Wrong answers use more tokens

Hypothesis: when a model is uncertain it produces longer reasoning chains and is more likely to reason itself into the wrong answer

This is visible in both variants and is independent of the repitition effect
* but informs `max_tokens` decision for the full run

In [5]:
correct_tokens   = [r["base_tokens"] for r in data if     r["base_correct"]]
incorrect_tokens = [r["base_tokens"] for r in data if not r["base_correct"]]

print("Token count by correctness (baseline):")
print(f"  Correct   — median={statistics.median(correct_tokens):.0f}  "
      f"mean={statistics.mean(correct_tokens):.0f}  "
      f"max={max(correct_tokens)}")
print(f"  Incorrect — median={statistics.median(incorrect_tokens):.0f}  "
      f"mean={statistics.mean(incorrect_tokens):.0f}  "
      f"max={max(incorrect_tokens)}")
print()

# repeat variant
correct_rep_tokens   = [r["rep_tokens"] for r in data if     r["rep_correct"]]
incorrect_rep_tokens = [r["rep_tokens"] for r in data if not r["rep_correct"]]

print("Token count by correctness (repeat):")
print(f"  Correct   — median={statistics.median(correct_rep_tokens):.0f}  "
      f"mean={statistics.mean(correct_rep_tokens):.0f}  "
      f"max={max(correct_rep_tokens)}")
print(f"  Incorrect — median={statistics.median(incorrect_rep_tokens):.0f}  "
      f"mean={statistics.mean(incorrect_rep_tokens):.0f}  "
      f"max={max(incorrect_rep_tokens)}")

Token count by correctness (baseline):
  Correct   — median=27  mean=39  max=129
  Incorrect — median=58  mean=60  max=129

Token count by correctness (repeat):
  Correct   — median=27  mean=38  max=129
  Incorrect — median=56  mean=56  max=129


### Latency Anomalies
Repeat much faster than baseline

In some questions the repeat variant generated far fewer tokens than baseline (therefore faster). This suggests the doubled prompt made the model more decisive.
* it committed to an answer quickly rather than reasoning at depth
* whether decisiveness helps or hurts accuracy is an open question at this sample size.

In [6]:
SPEEDUP_THRESHOLD = 0.6  # repeat latency < 60% of baseline latency

anomalies = [
    r for r in data
    if r["rep_latency_ms"] < r["base_latency_ms"] * SPEEDUP_THRESHOLD
]
anomalies.sort(key=lambda r: r["base_latency_ms"] - r["rep_latency_ms"], reverse=True)

print(f"Questions where repeat was <{SPEEDUP_THRESHOLD:.0%} of baseline latency: {len(anomalies)}")
print()
for r in anomalies:
    saving_ms = r["base_latency_ms"] - r["rep_latency_ms"]
    tok_delta = r["rep_tokens"] - r["base_tokens"]
    correct_str = f"base={' ✓' if r['base_correct'] else ' ✗'}  rep={' ✓' if r['rep_correct'] else ' ✗'}"
    print(f"  {r['question_id']:35}  "
          f"base={r['base_latency_ms']:.0f}ms/{r['base_tokens']}tok  "
          f"rep={r['rep_latency_ms']:.0f}ms/{r['rep_tokens']}tok  "
          f"saved={saving_ms:.0f}ms  Δtok={tok_delta:+d}  {correct_str}")

Questions where repeat was <60% of baseline latency: 5

  Mercury_188528                       base=6795ms/129tok  rep=1845ms/9tok  saved=4949ms  Δtok=-120  base= ✗  rep= ✗
  Mercury_7271758                      base=6898ms/129tok  rep=2423ms/15tok  saved=4475ms  Δtok=-114  base= ✗  rep= ✗
  TIMSS_2007_8_pg109                   base=3724ms/65tok  rep=1835ms/11tok  saved=1889ms  Δtok=-54  base= ✓  rep= ✓
  MCAS_2012_5_23614                    base=4419ms/72tok  rep=2565ms/14tok  saved=1854ms  Δtok=-58  base= ✓  rep= ✓
  Mercury_7172708                      base=3608ms/58tok  rep=2025ms/10tok  saved=1583ms  Δtok=-48  base= ✗  rep= ✗


## Summary & Action Items

| Finding | Impact | Fix before full run |
|---|---|---|
| `max_tokens=128` cap | 1 discordant result is an artifact | Raise to `256` in `MLXAgentConfig` |
| Numeric gold keys (3 NYSEDREGENTS Qs) | Hardcoded wrong for both variants; pollutes accuracy | Fix `arc_loader.py` to map `1/2/3/4 → A/B/C/D` |
| Verbosity = uncertainty signal | Wrong answers ~2× longer; cap hides reasoning | Confirmed by raising `max_tokens` |
| Repeat causes early cutoff in some Qs | Model more decisive but not necessarily more accurate | Worth tracking in full run |

**McNemar interpretation:** With 5 vs 1 discordant pairs the test statistic is ~2.67 (p ≈ 0.10), which is **not significant** at α=0.05. The -4% delta is a trend, not a finding. Removing the 1 cap-tainted discordant pair would give 4 vs 1, weakening the signal further. No conclusion about prompt repetition can be drawn from this pilot — which is exactly what a pilot is for.